# Firefighting Simulator — simulação de atuação do jato de água

Este notebook simula a atuação de atuadores responsáveis pelo direcionamento do jato de água sobre regiões de chama.

Ele foi adaptado para rodar dentro da estrutura portátil do repositório **BLAZE**, sem depender dos caminhos do servidor Jupyter da UFSC.

Os setups oficiais ficam em:

```text
BLAZE/jet_automation/parameters_setups/official/firefighting_simulator/
```

Os setups modificados pelo usuário são copiados/salvos localmente em:

```text
BLAZE/jet_automation/parameters_setups/user/firefighting_simulator/
```

A pasta `user/` é ignorada pelo GitHub, evitando alterações acidentais nos setups oficiais.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import cv2
import asyncio
import json
import os
import sys
import shutil
from pathlib import Path

# A renderização animada usa widgets.Image, evitando recriar figuras Matplotlib por frame.

def add_micro_hot_noise(T_base, rng, W, H, num_micro=22):
    micro = np.zeros_like(T_base)
    placed = []
    for _ in range(num_micro):
        for _try in range(60):
            cx = int(rng.integers(18, W - 18))
            cy = int(rng.integers(18, H - 18))
            ok = True
            for (px, py, prx, pry) in placed:
                if abs(cx - px) < (prx + 8) and abs(cy - py) < (pry + 8):
                    ok = False
                    break
            if ok:
                break
        rx = int(rng.integers(2, 5))
        ry = int(rng.integers(2, 5))
        angle = float(rng.uniform(0, 180))
        temp = float(rng.uniform(0.64, 0.78))
        cv2.ellipse(micro, (cx, cy), (rx, ry), angle, 0, 360, temp, -1)
        placed.append((cx, cy, rx, ry))
    micro = cv2.GaussianBlur(micro, (0, 0), sigmaX=0.9, sigmaY=0.9)
    return np.maximum(T_base, micro)

from IPython.display import display

# ==========================================================
# Dimensões e grade
# ==========================================================
H, W = 280, 460
Y, X = np.mgrid[0:H, 0:W]
TOTAL_PIXELS = H * W
MAX_STOP_PIXELS = TOTAL_PIXELS // 4
rng = np.random.default_rng(2026)
dt = 0.10
CENTROID_DWELL_FRAMES = 4  # valor padrão; o controle real vem do slider de permanência
COOL_RADIUS = 70            # raio padrão de atuação do cursor redutor
COOL_INTENSITY = 1.00      # intensidade padrão do resfriamento
RECOVERY_RATE_DEFAULT = 0.20  # 0 = mínima recuperação, 1 = recuperação mais rápida
OUTPUT_PANEL_WIDTH = 1280  # largura-alvo do painel com as duas imagens de resultado

# Centros dos dois fogos
main_cx, main_cy = W * 0.40, H * 0.66
side_cx, side_cy = W * 0.72, H * 0.71

# ==========================================================
# Estado global
# ==========================================================
cool_gray = np.zeros((H, W), dtype=np.float32)
cool_yellow = np.zeros((H, W), dtype=np.float32)


last_frame = {"value": 0}

state = {
    "x": 26,
    "y": H - 24,
    "last_move_frame": 0,
    "pause_until_frame": -1,
    "route": [],
    "route_index": 0,
    "mode": "borda",   # borda | centroides
    "movement_phase": "aproximação/entre trajetórias",
    "is_extinguished": False,
    "main_region": None,
    "secondary_region": None,
    "regions": [],
    "hot_pixels": 0,
    "frame": 0,
    "running": False,
    "animation_task": None,
}

# Estado da visualização persistente.
# A figura é criada apenas uma vez; depois atualizamos set_data/artistas.
# Isso evita o acúmulo de figuras e reduz o lag do frontend do Jupyter.
viz = {
    "initialized": False,
    "fig": None,
    "axes": None,
    "img_sem_filtro": None,
    "img_filtrada": None,
    "status_text": None,
    "dynamic_artists": [],
    "legend": None,
}

# ==========================================================
# Utilidades
# ==========================================================
def clip01(a):
    return np.clip(a, 0.0, 1.0)

def gaussian2d(x0, y0, sx, sy, amp):
    return amp * np.exp(-(((X - x0) ** 2) / (2 * sx ** 2) + ((Y - y0) ** 2) / (2 * sy ** 2)))

def move_point_towards(tx, ty, max_step=10):
    dx = tx - state["x"]
    dy = ty - state["y"]
    dist = float(np.hypot(dx, dy))
    if dist <= max_step or dist == 0:
        state["x"] = int(np.clip(round(tx), 0, W - 1))
        state["y"] = int(np.clip(round(ty), 0, H - 1))
        return True
    s = max_step / dist
    state["x"] = int(np.clip(round(state["x"] + dx * s), 0, W - 1))
    state["y"] = int(np.clip(round(state["y"] + dy * s), 0, H - 1))
    return False

def region_components(mask, min_area=15):
    mask_u8 = mask.astype(np.uint8)
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
    regs = []
    for idx in range(1, n_labels):
        area = int(stats[idx, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        x = int(stats[idx, cv2.CC_STAT_LEFT])
        y = int(stats[idx, cv2.CC_STAT_TOP])
        w = int(stats[idx, cv2.CC_STAT_WIDTH])
        h = int(stats[idx, cv2.CC_STAT_HEIGHT])
        comp = labels == idx
        regs.append({
            "label": idx,
            "mask": comp,
            "area": area,
            "cx": float(centroids[idx][0]),
            "cy": float(centroids[idx][1]),
            "bbox": (x, y, w, h),
            "bbox_prod": int(w * h),
        })
    regs.sort(key=lambda r: r["area"], reverse=True)
    return regs

def nearest_neighbor_route(points, start_xy):
    pts = [(int(p[0]), int(p[1])) for p in points]
    if not pts:
        return []
    unused = pts.copy()
    ordered = []
    current = (int(start_xy[0]), int(start_xy[1]))
    while unused:
        idx = int(np.argmin([np.hypot(current[0]-p[0], current[1]-p[1]) for p in unused]))
        nxt = unused.pop(idx)
        ordered.append(nxt)
        current = nxt
    return ordered

# ==========================================================
# Fundo e mapa de cor
# ==========================================================
def background_rgb():
    yy = np.linspace(0, 1, H).reshape(H, 1, 1)
    xx = np.linspace(0, 1, W).reshape(1, W, 1)
    if "blue_sky" == "blue_sky":
        top = np.array([0.02, 0.02, 0.04], dtype=np.float32).reshape(1,1,3)
        mid = np.array([0.10, 0.16, 0.30], dtype=np.float32).reshape(1,1,3)
        low = np.array([0.18, 0.28, 0.52], dtype=np.float32).reshape(1,1,3)
        split = 0.36
        bg = np.where(
            yy < split,
            top*(1 - yy/split) + mid*(yy/split),
            mid*(1 - (yy-split)/(1-split)) + low*((yy-split)/(1-split))
        )
        bg = bg + 0.02*np.sin(8*xx + 5*yy)
        return clip01(bg)
    else:
        top = np.array([0.12, 0.12, 0.13], dtype=np.float32).reshape(1,1,3)
        low = np.array([0.18, 0.18, 0.19], dtype=np.float32).reshape(1,1,3)
        bg = top*(1-yy) + low*yy
        return clip01(np.repeat(bg, W, axis=1))

BG_RGB = background_rgb()


def aplicar_mistura(base_rgb, mask, target_color, alpha):
    """Mistura uma cor-alvo em pixels selecionados com transparência alpha."""
    if np.isscalar(alpha):
        alpha_arr = float(alpha) * mask.astype(np.float32)
    else:
        alpha_arr = np.clip(alpha.astype(np.float32), 0.0, 1.0) * mask.astype(np.float32)
    alpha_arr = alpha_arr[..., None]
    target = np.array(target_color, dtype=np.float32).reshape(1, 1, 3)
    return clip01(base_rgb * (1.0 - alpha_arr) + target * alpha_arr)


def ajustar_visual_componentes(rgb, T_ref, limiar_componente=0.12):
    """
    Renderiza cada componente com lógica visual dependente da área:
    - todas as chamas têm bordas amarelas e região interna laranja;
    - somente chamas suficientemente grandes ganham núcleo vermelho;
    - o vermelho diminui conforme a área total da chama reduz;
    - tons mais escuros representam regiões mais quentes.
    """
    mask = (T_ref >= float(limiar_componente)).astype(np.uint8)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)

    AREA_RED_MIN = 260.0
    AREA_RED_FULL = 2200.0

    for idx in range(1, n_labels):
        area = int(stats[idx, cv2.CC_STAT_AREA])
        if area < 6:
            continue

        x = int(stats[idx, cv2.CC_STAT_LEFT])
        y = int(stats[idx, cv2.CC_STAT_TOP])
        w = int(stats[idx, cv2.CC_STAT_WIDTH])
        h = int(stats[idx, cv2.CC_STAT_HEIGHT])

        comp = (labels[y:y+h, x:x+w] == idx).astype(np.uint8)
        T_patch = T_ref[y:y+h, x:x+w].astype(np.float32)
        if comp.sum() == 0:
            continue

        dist = cv2.distanceTransform(comp, cv2.DIST_L2, 5).astype(np.float32)
        max_dist = float(dist.max())
        if max_dist <= 1e-6:
            continue

        dnorm = dist / max_dist
        temp_norm = np.clip((T_patch - 0.10) / 0.90, 0.0, 1.0)
        area_norm = np.clip((area - AREA_RED_MIN) / (AREA_RED_FULL - AREA_RED_MIN), 0.0, 1.0)

        patch_rgb = rgb[y:y+h, x:x+w].copy()

        # Halo amarelo periférico para todas as chamas.
        yellow_mask = comp.astype(bool)
        yellow_alpha = np.clip(0.32 + 0.34 * (1.0 - dnorm) + 0.10 * (1.0 - temp_norm), 0.0, 0.72)
        patch_rgb = aplicar_mistura(patch_rgb, yellow_mask, [1.00, 0.86, 0.14], yellow_alpha)

        # Núcleo laranja presente em toda chama, mais forte no interior.
        orange_mask = yellow_mask
        orange_alpha = np.clip(((dnorm - 0.16) / 0.48), 0.0, 1.0)
        orange_alpha *= np.clip(0.52 + 0.42 * temp_norm, 0.0, 1.0)

        orange_dark = np.array([0.82, 0.22, 0.01], dtype=np.float32)
        orange_light = np.array([0.98, 0.55, 0.08], dtype=np.float32)
        orange_color_map = orange_light.reshape(1,1,3) * (1.0 - temp_norm[...,None]) + orange_dark.reshape(1,1,3) * temp_norm[...,None]
        alpha_arr = orange_alpha[..., None] * orange_mask.astype(np.float32)[..., None]
        patch_rgb = clip01(patch_rgb * (1.0 - alpha_arr) + orange_color_map * alpha_arr)

        # Núcleo vermelho apenas quando a chama é suficientemente grande.
        if area_norm > 0.0:
            red_start = 0.84 - 0.42 * area_norm
            red_span = max(0.10, 0.24 - 0.08 * area_norm)
            red_alpha = np.clip((dnorm - red_start) / red_span, 0.0, 1.0)
            red_alpha *= np.clip(0.55 + 0.45 * temp_norm, 0.0, 1.0)

            red = np.array([0.72, 0.04, 0.00], dtype=np.float32)
            red_dark = np.array([0.34, 0.00, 0.00], dtype=np.float32)
            red_color_map = red.reshape(1,1,3) * (1.0 - temp_norm[...,None]) + red_dark.reshape(1,1,3) * temp_norm[...,None]
            alpha_arr = (red_alpha * area_norm)[..., None] * orange_mask.astype(np.float32)[..., None]
            patch_rgb = clip01(patch_rgb * (1.0 - alpha_arr) + red_color_map * alpha_arr)

        rgb[y:y+h, x:x+w] = np.where(comp[...,None].astype(bool), patch_rgb, rgb[y:y+h, x:x+w])

    return clip01(rgb)


def thermal_colormap_custom(T):
    T = clip01(T).astype(np.float32)
    heat = cv2.GaussianBlur(T, (0, 0), 1.1).astype(np.float32)
    rgb = BG_RGB.astype(np.float32).copy()

    # Aura geral quente em volta das regiões com calor.
    warm = np.array([0.96, 0.44, 0.08], dtype=np.float32).reshape(1, 1, 3)
    halo = np.array([1.00, 0.78, 0.18], dtype=np.float32).reshape(1, 1, 3)

    w0 = np.clip((heat - 0.05) / 0.14, 0.0, 1.0)[..., None]
    w1 = np.clip((heat - 0.12) / 0.18, 0.0, 1.0)[..., None]
    rgb = (1.0 - w0) * rgb + w0 * (0.72 * rgb + 0.28 * halo)
    rgb = (1.0 - 0.55 * w1) * rgb + (0.55 * w1) * warm

    # Ajuste principal da cor por componente/área.
    rgb = ajustar_visual_componentes(rgb, heat, limiar_componente=0.12)

    return clip01(cv2.GaussianBlur(rgb.astype(np.float32), (0, 0), 0.8))

# ==========================================================
# Construção do campo térmico
# ==========================================================
def flame_shape(frame, cx, cy, scale=1.0, side=False):
    t = frame * dt
    sway = 7*np.sin(0.55*t + (0.8 if side else 0.0))

    # corpo principal largo embaixo e afunilado em cima
    T = np.zeros((H,W), dtype=np.float32)
    T += gaussian2d(cx, cy + 2, 34*scale, 23*scale, 0.86)
    T += gaussian2d(cx - 18*scale, cy + 2, 16*scale, 12*scale, 0.24)
    T += gaussian2d(cx + 18*scale, cy + 2, 16*scale, 12*scale, 0.24)

    # halo mais largo lateralmente
    T += gaussian2d(cx, cy - 6, 58*scale, 40*scale, 0.28)
    T += gaussian2d(cx - 24*scale, cy - 5, 22*scale, 18*scale, 0.10)
    T += gaussian2d(cx + 24*scale, cy - 5, 22*scale, 18*scale, 0.10)

    # pluma vertical disforme
    for k in range(6):
        yk = cy - 20*scale - 18*scale*k
        xk = cx + sway*(0.32 + 0.10*k) + 5*np.sin(0.7*t + 0.9*k)
        sx = max(7*scale, (18 - 2*k)*scale)
        sy = max(12*scale, (24 - 2*k)*scale)
        amp = max(0.08, 0.24 - 0.028*k) * (0.95 if not side else 0.85)
        T += gaussian2d(xk, yk, sx, sy, amp)

    # topo destacado e chama desconectada pequena
    T += gaussian2d(cx + 0.5*sway, cy - 95*scale, 9*scale, 13*scale, 0.22*(0.9 if not side else 0.75))
    if not side:
        T += gaussian2d(cx - 10 + 4*np.sin(t), cy - 118, 11, 15, 0.16)
        T += gaussian2d(cx - 9 + 4*np.sin(t), cy - 118, 5, 7, 0.12)
    return T


# ruídos em pequenos focos borrados, com formatos variados e sem sobreposição
def build_noise_maps():
    hot = np.zeros((H, W), dtype=np.float32)
    warm = np.zeros((H, W), dtype=np.float32)
    centers = []
    tries = 0

    def place_ok(x0, y0, reach):
        if np.hypot(x0 - main_cx, y0 - main_cy) < 120:
            return False
        if np.hypot(x0 - side_cx, y0 - side_cy) < 95:
            return False
        if any(np.hypot(x0 - x1, y0 - y1) < (reach + r1) for x1, y1, r1 in centers):
            return False
        return True

    while len(centers) < 11 and tries < 5000:
        tries += 1
        x0 = int(rng.integers(36, W - 36))
        y0 = int(rng.integers(32, H - 36))
        reach = int(rng.integers(18, 28))
        if not place_ok(x0, y0, reach):
            continue

        kind = "hot" if len(centers) < 4 else "warm"
        shape_id = int(rng.integers(0, 4))
        blob = np.zeros((H, W), dtype=np.float32)

        if shape_id == 0:
            # mancha elíptica alongada
            sx = float(rng.uniform(10, 18))
            sy = float(rng.uniform(5, 10))
            theta = float(rng.uniform(0, np.pi))
            xr = (X - x0) * np.cos(theta) + (Y - y0) * np.sin(theta)
            yr = -(X - x0) * np.sin(theta) + (Y - y0) * np.cos(theta)
            blob = np.exp(-((xr**2) / (2 * sx**2) + (yr**2) / (2 * sy**2)))
        elif shape_id == 1:
            # formato em L suave
            blob = np.maximum(
                np.exp(-(((X - x0)**2) / (2 * 13.0**2) + ((Y - y0)**2) / (2 * 5.5**2))),
                np.exp(-(((X - (x0 - 8))**2) / (2 * 5.5**2) + ((Y - (y0 + 8))**2) / (2 * 13.0**2)))
            )
        elif shape_id == 2:
            # mancha com ondulações
            for k in range(3):
                dx = float(rng.uniform(-10, 10))
                dy = float(rng.uniform(-8, 8))
                sx = float(rng.uniform(6, 12))
                sy = float(rng.uniform(6, 11))
                blob = np.maximum(blob, np.exp(-(((X - (x0 + dx))**2) / (2 * sx**2) + ((Y - (y0 + dy))**2) / (2 * sy**2))))
        else:
            # gota horizontal delgada
            blob = np.exp(-(((X - x0)**2) / (2 * 15.0**2) + ((Y - y0)**2) / (2 * 6.5**2)))

        blob = cv2.GaussianBlur(blob.astype(np.float32), (0, 0), 2.6)

        if kind == "hot":
            hot = np.maximum(hot, blob * float(rng.uniform(0.64, 0.80)))
        else:
            warm = np.maximum(warm, blob * float(rng.uniform(0.44, 0.56)))

        centers.append((x0, y0, reach))

    return hot, warm

NOISE_HOT, NOISE_WARM = build_noise_maps()

# Máscaras de focos pequenos.
# A versão anterior usava apenas NOISE_HOT >= 0.60 e min_area=25.
# Isso fazia alguns focos visíveis não entrarem nas regras de colapso/apagamento.
NOISE_HOT_MASKS = [r["mask"] for r in region_components(NOISE_HOT >= 0.50, min_area=8)]
NOISE_WARM_MASKS = [r["mask"] for r in region_components(NOISE_WARM >= 0.42, min_area=8)]


def build_visual_speckles(num_speckles=42):
    """
    Gera chuviscos fixos vermelho-alaranjados.

    - `heat_map` participa apenas da visualização/pré-processamento, para que
      os chuviscos possam sobreviver quando o filtro for pouco eficiente;
    - `color` e `alpha` mantêm a poluição visual explícita na imagem sem filtro.
    """
    speckle_rng = np.random.default_rng(3341)
    alpha = np.zeros((H, W), dtype=np.float32)
    color = np.zeros((H, W, 3), dtype=np.float32)
    heat_map = np.zeros((H, W), dtype=np.float32)

    for _ in range(num_speckles):
        cx = int(speckle_rng.integers(14, W - 14))
        cy = int(speckle_rng.integers(12, H - 12))
        if np.hypot(cx - main_cx, cy - main_cy) < 55 or np.hypot(cx - side_cx, cy - side_cy) < 40:
            continue

        rx = int(speckle_rng.integers(1, 4))
        ry = int(speckle_rng.integers(1, 3))
        ang = float(speckle_rng.uniform(0, 180))
        local_alpha = float(speckle_rng.uniform(0.45, 0.78))
        local_heat = float(speckle_rng.uniform(0.48, 0.64))
        local_color = np.array([
            float(speckle_rng.uniform(0.85, 1.00)),
            float(speckle_rng.uniform(0.22, 0.46)),
            float(speckle_rng.uniform(0.00, 0.08)),
        ], dtype=np.float32)

        mask = np.zeros((H, W), dtype=np.float32)
        cv2.ellipse(mask, (cx, cy), (rx, ry), ang, 0, 360, 1.0, -1)
        mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=0.55, sigmaY=0.55)

        a = np.clip(mask * local_alpha, 0.0, 1.0)
        update = a > alpha
        alpha[update] = a[update]
        color[update] = local_color

        heat_blob = cv2.GaussianBlur((mask * local_heat).astype(np.float32), (0, 0), sigmaX=0.45, sigmaY=0.45)
        heat_map = np.maximum(heat_map, heat_blob)

    return heat_map.astype(np.float32), color, alpha[..., None]


VISUAL_SPECKLE_HEAT, VISUAL_SPECKLE_COLOR, VISUAL_SPECKLE_ALPHA = build_visual_speckles()

def aplicar_chuviscos_visuais(rgb):
    """Aplica os chuviscos somente na visualização sem filtro."""
    return clip01((1.0 - VISUAL_SPECKLE_ALPHA) * rgb + VISUAL_SPECKLE_ALPHA * VISUAL_SPECKLE_COLOR)


def heart_seed_mask(cx, cy, scale=1.0):
    heart = gaussian2d(cx, cy + 4, 20*scale, 13*scale, 1.0)
    heart += gaussian2d(cx - 8*scale, cy + 5, 8*scale, 6*scale, 0.25)
    heart += gaussian2d(cx + 8*scale, cy + 5, 8*scale, 6*scale, 0.25)
    return heart > 0.42

# Máscaras fixas pré-computadas para evitar recálculo a cada frame.
MAIN_HEART_MASK = heart_seed_mask(main_cx, main_cy, scale=1.18)
SIDE_HEART_MASK = heart_seed_mask(side_cx, side_cy, scale=0.76)

# O fogo principal fica estático. A redução visual vem do resfriamento e das
# regras de colapso, não de uma chama se deslocando a todo frame.
MAIN_STATIC_FIRE = flame_shape(0, main_cx, main_cy, scale=1.18, side=False).astype(np.float32)
SIDE_STATIC_FIRE = flame_shape(17, side_cx, side_cy, scale=0.76, side=True).astype(np.float32)

def component_survival_factor(base_component, cooled_component, heart_mask=None):
    if heart_mask is None:
        heart_mask = base_component > max(0.55, 0.72 * float(base_component.max()))
    if not np.any(heart_mask):
        return 0.0
    base_mean = float(np.mean(base_component[heart_mask])) + 1e-6
    cooled_mean = float(np.mean(cooled_component[heart_mask]))
    frac = np.clip(cooled_mean / base_mean, 0.0, 1.0)
    return float(frac ** 2.0)


def construir_cena_termica(frame):
    bg = np.zeros((H,W), dtype=np.float32)
    # Focos principais estáticos: menos custo e menos oscilação visual.
    main = MAIN_STATIC_FIRE
    side = SIDE_STATIC_FIRE

    T = bg + main + side
    T = np.maximum(T, NOISE_HOT)
    T = np.maximum(T, NOISE_WARM)
    T = cv2.GaussianBlur(T.astype(np.float32), (0,0), 2.1)

    parts = {
        "main": main.astype(np.float32),
        "side": side.astype(np.float32),
        "noise_hot": NOISE_HOT.astype(np.float32),
        "warm": NOISE_WARM.astype(np.float32),
        "main_heart_mask": MAIN_HEART_MASK,
        "side_heart_mask": SIDE_HEART_MASK,
        "noise_hot_masks": NOISE_HOT_MASKS,
        "noise_warm_masks": NOISE_WARM_MASKS,
        "noise_masks": NOISE_HOT_MASKS + NOISE_WARM_MASKS,
    }
    return clip01(T), parts

def atualizar_resfriamento(frame, px, py, effect_radius=COOL_RADIUS, intensity=COOL_INTENSITY, recovery_rate=RECOVERY_RATE_DEFAULT):
    delta = max(0, frame - last_frame["value"])
    last_frame["value"] = frame
    if delta == 0:
        return

    effect_radius = float(max(8.0, effect_radius))
    intensity = float(max(0.0, intensity))
    recovery_rate = float(np.clip(recovery_rate, 0.0, 1.0))

    raio = int(round(effect_radius))
    x0 = max(0, int(round(px)) - raio)
    x1 = min(W, int(round(px)) + raio + 1)
    y0 = max(0, int(round(py)) - raio)
    y1 = min(H, int(round(py)) + raio + 1)

    Xs = X[y0:y1, x0:x1]
    Ys = Y[y0:y1, x0:x1]
    D2 = (Xs - px)**2 + (Ys - py)**2

    sigma_core = max(3.5, 0.12 * effect_radius)
    sigma_near = max(sigma_core + 2.0, 0.24 * effect_radius)
    sigma_wide = max(sigma_near + 3.0, 0.42 * effect_radius)

    core = np.exp(-D2 / (2 * sigma_core**2))
    near = np.exp(-D2 / (2 * sigma_near**2))
    wide = np.exp(-D2 / (2 * sigma_wide**2))

    gain_core = intensity * (0.082 * delta)
    gain_near = intensity * (0.020 * delta)
    gain_wide = intensity * (0.030 * delta)

    cool_gray[y0:y1, x0:x1] += gain_core * core + gain_near * np.maximum(near - 0.22, 0.0)
    cool_yellow[y0:y1, x0:x1] += gain_wide * np.maximum(wide - 0.16, 0.0)

    # recuperação: valores maiores dissipam mais rápido o efeito do resfriamento,
    # favorecendo a reconstituição/espalhamento visual da chama.
    retain_gray = float(np.clip(0.9990 - 0.0040 * recovery_rate, 0.9920, 0.9995))
    retain_yellow = float(np.clip(0.9978 - 0.0060 * recovery_rate, 0.9880, 0.9990))

    cool_gray[:] *= retain_gray
    cool_yellow[:] *= retain_yellow
    cool_gray[:] = np.minimum(cool_gray, 1.25)
    cool_yellow[:] = np.minimum(cool_yellow, 0.82)

def aplicar_regras(T, parts, recovery_rate=RECOVERY_RATE_DEFAULT):
    cooled = T.copy()

    # redução global local da temperatura para encolher o foco inteiro ao redor do jato
    local_cool = np.clip(0.85 * cool_gray + 0.35 * cool_yellow, 0.0, 1.0)
    cooled *= (1.0 - 0.58 * np.clip(local_cool - 0.05, 0.0, 1.0))

    # regiões próximas ao jato tendem a migrar para tons mais amenos
    yellow_target = 0.49
    yellow_w = np.clip(cool_yellow - 0.10, 0.0, 0.62)
    cooled -= yellow_w * np.maximum(cooled - yellow_target, 0.0)

    # centro diretamente atingido esfria ainda mais
    dark_target = 0.03
    dark_w = np.clip(cool_gray - 0.16, 0.0, 1.0)
    cooled -= dark_w * np.maximum(cooled - dark_target, 0.0)

    cooled = cv2.GaussianBlur(cooled.astype(np.float32), (0,0), 0.8)

    # ------------------------------------------------------
    # A chama externa depende do coração: quando o vermelho
    # do coração some, a aura e a pluma desse foco colapsam.
    # ------------------------------------------------------
    main_factor = component_survival_factor(parts["main"], cooled, parts["main_heart_mask"])
    side_factor = component_survival_factor(parts["side"], cooled, parts["side_heart_mask"])

    main_support = parts["main"] > 0.06
    side_support = parts["side"] > 0.06
    cooled[main_support] *= main_factor
    cooled[side_support] *= side_factor

    # pequenos focos quentes: cada um colapsa com seu próprio "coração"
    for mask in parts["noise_hot_masks"]:
        base_comp = parts["noise_hot"] * mask.astype(np.float32)
        comp_factor = component_survival_factor(base_comp, cooled, heart_mask=base_comp > 0.58)
        support = base_comp > 0.04
        cooled[support] *= comp_factor

    # pequenos focos mornos: agora também respondem ao jato.
    # Antes, alguns focos visíveis eram recolocados depois do resfriamento.
    for mask in parts["noise_warm_masks"]:
        base_comp = parts["warm"] * mask.astype(np.float32)
        comp_factor = component_survival_factor(base_comp, cooled, heart_mask=base_comp > 0.43)
        support = base_comp > 0.03
        cooled[support] *= comp_factor
    # Focos mornos permanecem como contexto visual fraco, mas não são mais imunes.
    residual_gain = 0.82 + 0.26 * float(np.clip(recovery_rate, 0.0, 1.0))
    warm_residual = parts["warm"] * residual_gain
    warm_residual *= 1.0 - np.clip(1.6 * cool_gray + 0.7 * cool_yellow, 0.0, 1.0)
    cooled = np.maximum(cooled, warm_residual)

    cooled[cooled < 0.010] = 0.0
    return clip01(cooled)

def gerar_campo_termico_atual(frame, recovery_rate=RECOVERY_RATE_DEFAULT):
    T, parts = construir_cena_termica(frame)
    return aplicar_regras(T, parts, recovery_rate=recovery_rate)

def preprocess_hot_regions(T, thr_hot=0.58, erosion_size=3, erosion_iters=1, min_area=12):
    raw_hot = (T >= float(thr_hot)).astype(np.uint8)

    erosion_size = max(1, int(erosion_size))
    if erosion_size % 2 == 0:
        erosion_size += 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erosion_size, erosion_size))

    filtered = raw_hot.copy()
    if erosion_iters > 0:
        filtered = cv2.erode(filtered, kernel, iterations=int(erosion_iters))
        filtered = cv2.dilate(filtered, kernel, iterations=int(erosion_iters))

    raw_regions = region_components(raw_hot > 0, min_area=min_area)
    regions = region_components(filtered > 0, min_area=min_area)

    main_region = regions[0] if regions else None
    secondary_region = regions[1] if len(regions) >= 2 else None
    return raw_hot > 0, filtered > 0, raw_regions, regions, main_region, secondary_region

# ==========================================================
# Estratégias de trajetória
# ==========================================================
def build_bottom_route(mask, n_points=12, y_offset=4):
    ys, xs = np.where(mask)
    if xs.size == 0:
        return []
    x_left, x_right = int(xs.min()), int(xs.max())
    anchors_x = np.linspace(x_left, x_right, max(2, int(n_points)))
    route = []
    used = set()
    y_offset = max(0, int(y_offset))

    for xq in anchors_x:
        xq = int(round(xq))
        best = None
        best_key = None
        for dx in range(-12, 13):
            xc = int(np.clip(xq + dx, 0, W - 1))
            ys_col = np.where(mask[:, xc])[0]
            if ys_col.size == 0:
                continue
            y_bottom = int(ys_col.max())
            y_target = y_bottom
            if y_offset > 0:
                ys_valid = ys_col[ys_col <= y_bottom - y_offset]
                if ys_valid.size > 0:
                    y_target = int(ys_valid.max())
            key = (abs(dx), -y_bottom)
            if best_key is None or key < best_key:
                best = (xc, y_target)
                best_key = key
        if best is not None and best not in used:
            route.append(best)
            used.add(best)

    # compactação
    compact = []
    for p in route:
        if not compact or np.hypot(p[0]-compact[-1][0], p[1]-compact[-1][1]) > 5:
            compact.append(p)
    if len(compact) <= 1:
        return compact

    p_first, p_last = compact[0], compact[-1]
    d_first = np.hypot(state["x"]-p_first[0], state["y"]-p_first[1])
    d_last = np.hypot(state["x"]-p_last[0], state["y"]-p_last[1])
    return compact if d_first <= d_last else list(reversed(compact))

def build_centroid_route(regions):
    points = [(int(round(r["cx"])), int(round(r["cy"]))) for r in regions]
    return nearest_neighbor_route(points, (state["x"], state["y"]))


def dist_cursor_to_point(p):
    return float(np.hypot(state["x"] - p[0], state["y"] - p[1]))


def choose_nearest_offset_route(large_regions, n_points=12, y_offset=4):
    """
    Mantém a prioridade por regiões grandes aptas à estratégia de offset,
    mas escolhe a próxima trajetória pela menor distância até o cursor.
    """
    candidates = []
    for reg in large_regions:
        route = build_bottom_route(reg["mask"], n_points=n_points, y_offset=y_offset)
        if not route:
            continue
        d0 = dist_cursor_to_point(route[0])
        candidates.append((d0, reg, route))
    if not candidates:
        return None, []
    candidates.sort(key=lambda item: item[0])
    return candidates[0][1], candidates[0][2]


def choose_route(T, thr_hot, erosion_size, erosion_iters, n_points, y_offset, bbox_prod_min):
    raw_hot, filtered_hot, raw_regions, regions, main_region, secondary_region = preprocess_hot_regions(
        T, thr_hot=thr_hot, erosion_size=erosion_size, erosion_iters=erosion_iters, min_area=10
    )
    state["regions"] = regions
    state["main_region"] = main_region
    state["secondary_region"] = secondary_region

    # 1) Prioridade continua sendo das regiões que permitem trajetória por offset.
    # 2) Dentro desse grupo, a próxima trajetória escolhida é a mais próxima do cursor,
    #    não necessariamente a maior região.
    large_regions = [r for r in regions if r["bbox_prod"] >= int(bbox_prod_min)]
    chosen_region, route = choose_nearest_offset_route(
        large_regions, n_points=n_points, y_offset=y_offset
    )

    if route:
        state["mode"] = "borda"
    else:
        state["mode"] = "centroides"
        # Sem região grande apta ao offset, usa os focos fragmentados.
        # A ordem dos centróides já é por vizinho mais próximo a partir do cursor.
        route_regs = regions if len(regions) > 0 else raw_regions
        route = build_centroid_route(route_regs)

    state["route"] = route
    state["route_index"] = 0

def maybe_rebuild_route(frame, thr_hot, erosion_size, erosion_iters, n_points, y_offset, bbox_prod_min, recovery_rate=RECOVERY_RATE_DEFAULT):
    if state["is_extinguished"]:
        return
    finished = state["route_index"] >= len(state["route"])
    no_route = len(state["route"]) == 0
    if no_route or (finished and frame >= state["pause_until_frame"]):
        # Gera T somente quando a rota precisa ser reconstruída.
        # Isso evita uma segunda reconstrução completa do campo térmico em todo frame.
        T_route = gerar_campo_termico_atual(frame, recovery_rate=recovery_rate)
        choose_route(T_route, thr_hot, erosion_size, erosion_iters, n_points, y_offset, bbox_prod_min)


def update_motion(frame, border_speed=10, other_speed=16, dwell_frames=4):
    if state["is_extinguished"]:
        return
    if frame < state["pause_until_frame"]:
        return
    if state["route_index"] >= len(state["route"]):
        return

    # Velocidade por situação:
    # - vel. outras: aproximação até o primeiro ponto de qualquer nova trajetória
    #   e deslocamento entre focos/rotas fragmentadas;
    # - vel. borda: apenas enquanto percorre os pontos internos de uma trajetória
    #   gerada por offset de borda, isto é, depois que o primeiro ponto foi alcançado.
    if state["mode"] == "borda" and state["route_index"] > 0:
        eff_step = int(border_speed)
        state["movement_phase"] = "offset"
    else:
        eff_step = int(other_speed)
        state["movement_phase"] = "aproximação/entre trajetórias"
    eff_step = max(1, eff_step)

    tx, ty = state["route"][state["route_index"]]
    arrived = move_point_towards(tx, ty, max_step=eff_step)
    state["last_move_frame"] = frame

    if arrived:
        state["route_index"] += 1
        dwell_frames = max(0, int(dwell_frames))

        # Mantém o cursor sobre cada ponto por alguns frames antes de mudar o alvo.
        if dwell_frames > 0:
            state["pause_until_frame"] = max(state["pause_until_frame"], frame + dwell_frames)

        if state["route_index"] >= len(state["route"]):
            state["pause_until_frame"] = max(state["pause_until_frame"], frame + max(1, dwell_frames))

# ==========================================================
# Critério de parada

# ==========================================================
def stop_condition_hot_pixels(T, thr_stop=0.58, stop_pixels=0):
    hot_pixels = int(np.count_nonzero(T >= float(thr_stop)))
    return hot_pixels <= int(stop_pixels), hot_pixels

# ==========================================================
# Desenho leve com widgets.Image
# ==========================================================
def montar_frame(frame, vel_borda, vel_outras, dwell_frames, thr_hot, erosion_size, erosion_iters, stop_pixels,
                 n_points, thr_stop, y_offset, bbox_prod_min, cool_radius, cool_intensity, recovery_rate):
    """Executa a lógica física/segmentação e devolve as imagens e metadados do frame."""
    maybe_rebuild_route(frame, thr_hot, erosion_size, erosion_iters, n_points, y_offset, bbox_prod_min, recovery_rate=recovery_rate)
    update_motion(frame, border_speed=vel_borda, other_speed=vel_outras, dwell_frames=dwell_frames)

    if not state["is_extinguished"]:
        atualizar_resfriamento(frame, state["x"], state["y"], effect_radius=cool_radius, intensity=cool_intensity, recovery_rate=recovery_rate)

    T = gerar_campo_termico_atual(frame, recovery_rate=recovery_rate)
    T_visual = np.maximum(T, VISUAL_SPECKLE_HEAT)
    raw_hot, filtered_hot, raw_regions, regions, main_region, secondary_region = preprocess_hot_regions(
        T_visual, thr_hot=thr_hot, erosion_size=erosion_size, erosion_iters=erosion_iters, min_area=10
    )

    criterio_atingido, hot_pixels = stop_condition_hot_pixels(T, thr_stop=thr_stop, stop_pixels=stop_pixels)
    state["hot_pixels"] = hot_pixels
    state["regions"] = regions
    state["main_region"] = main_region
    state["secondary_region"] = secondary_region

    rgb_visual = thermal_colormap_custom(T_visual)
    rgb_sem_filtro = aplicar_chuviscos_visuais(rgb_visual)
    filtered_view = BG_RGB.copy()
    filtered_view[filtered_hot] = rgb_visual[filtered_hot]

    status = (
        f"frame = {int(frame)} | "
        f"modo = {state['mode']} | "
        f"fase = {state.get('movement_phase', '-')} | "
        f"vel borda = {int(vel_borda)} | "
        f"vel outras = {int(vel_outras)} | "
        f"permanência = {int(dwell_frames)} frames | "
        f"pixels acima do limiar = {hot_pixels} | "
        f"limiar = {int(stop_pixels)} px | "
        f"thr quente = {float(thr_hot):.2f} | "
        f"thr parada = {float(thr_stop):.2f} | "
        f"raio = {int(cool_radius)} | "
        f"intensidade = {float(cool_intensity):.2f} | "
        f"recuperação = {float(recovery_rate):.2f} | "
        f"prod. bbox mín. = {int(bbox_prod_min)}"
    )
    if criterio_atingido:
        status += " | critério de extinção atingido"
    status += " | executando" if state["running"] else " | parado pelo botão Stop"

    return rgb_sem_filtro, filtered_view, regions, list(state["route"]), status


def rgb_float_to_u8(img):
    return np.clip(img * 255.0, 0, 255).astype(np.uint8)


def codificar_jpeg_rgb(img_rgb_u8, qualidade=82):
    """Codifica uma imagem RGB uint8 para JPEG. Mais leve que redesenhar Matplotlib."""
    img_bgr = cv2.cvtColor(img_rgb_u8, cv2.COLOR_RGB2BGR)
    ok, buf = cv2.imencode(".jpg", img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(qualidade)])
    if not ok:
        raise RuntimeError("Falha ao codificar o frame em JPEG.")
    return buf.tobytes()


def montar_painel_visual(rgb_sem_filtro, filtered_view, regions, route, cool_radius):
    """Monta lado a lado as duas imagens e desenha trajetória/cursor com OpenCV."""
    left = rgb_float_to_u8(rgb_sem_filtro)
    right = rgb_float_to_u8(filtered_view).copy()

    # centróides das regiões
    for i, reg in enumerate(regions):
        color = (46, 107, 255) if i == 0 else (214, 214, 214)  # RGB
        cx = int(round(reg["cx"]))
        cy = int(round(reg["cy"]))
        cv2.circle(right, (cx, cy), 2, color, -1, lineType=cv2.LINE_AA)
        cv2.circle(right, (cx, cy), 3, (0, 0, 0), 1, lineType=cv2.LINE_AA)

    # rota
    if len(route) >= 2:
        pts = np.array([[int(p[0]), int(p[1])] for p in route], dtype=np.int32)
        cv2.polylines(right, [pts], isClosed=False, color=(68, 204, 102), thickness=1, lineType=cv2.LINE_AA)
        for p in pts:
            cv2.circle(right, tuple(p), 2, (0, 170, 51), -1, lineType=cv2.LINE_AA)
    elif len(route) == 1:
        p = (int(route[0][0]), int(route[0][1]))
        cv2.circle(right, p, 3, (0, 170, 51), -1, lineType=cv2.LINE_AA)

    # cursor/jato
    if not state["is_extinguished"]:
        x = int(state["x"])
        y = int(state["y"])
        cv2.circle(right, (x, y), max(4, int(round(cool_radius))), (120, 255, 120), 1, lineType=cv2.LINE_AA)
        cv2.rectangle(right, (x - 6, y - 6), (x + 6, y + 6), (0, 255, 0), 2, lineType=cv2.LINE_AA)

    title_h = 30
    gap = 8
    canvas = np.zeros((H + title_h, 2 * W + gap, 3), dtype=np.uint8)
    canvas[:title_h, :, :] = np.array([18, 18, 22], dtype=np.uint8)
    canvas[title_h:, :W, :] = left
    canvas[title_h:, W + gap:, :] = right

    cv2.putText(canvas, "Imagem termica sem filtro + chuviscos", (8, 21),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (235, 235, 235), 1, cv2.LINE_AA)
    cv2.putText(canvas, "Imagem final filtrada + trajetoria", (W + gap + 8, 21),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (235, 235, 235), 1, cv2.LINE_AA)

    # Amplia o painel para ocupar melhor a largura do notebook.
    # O conteúdo original continua lado a lado; apenas a escala visual é aumentada.
    if OUTPUT_PANEL_WIDTH is not None and OUTPUT_PANEL_WIDTH > canvas.shape[1]:
        escala = OUTPUT_PANEL_WIDTH / float(canvas.shape[1])
        nova_altura = int(round(canvas.shape[0] * escala))
        canvas = cv2.resize(canvas, (int(OUTPUT_PANEL_WIDTH), nova_altura), interpolation=cv2.INTER_LINEAR)

    return canvas


def desenhar(frame, vel_borda, vel_outras, dwell_frames, thr_hot, erosion_size, erosion_iters, stop_pixels,
             n_points, thr_stop, y_offset, bbox_prod_min, cool_radius, cool_intensity, recovery_rate):
    rgb_sem_filtro, filtered_view, regions, route, status = montar_frame(
        frame, vel_borda, vel_outras, dwell_frames, thr_hot, erosion_size, erosion_iters, stop_pixels,
        n_points, thr_stop, y_offset, bbox_prod_min, cool_radius, cool_intensity, recovery_rate
    )
    painel = montar_painel_visual(rgb_sem_filtro, filtered_view, regions, route, cool_radius)
    vis_image.value = codificar_jpeg_rgb(painel, qualidade=82)
    render_status.value = f"<b>Status:</b> {status}"

# ==========================================================
# Widgets e execução contínua
# ==========================================================
desc = widgets.HTML(
    value=f"""
    <div style="font-size:12px; line-height:1.35; max-width:1280px;">
    <b>Parâmetros dos widgets</b><br>
    <b>Iniciar</b>: começa/continua a execução em laço contínuo, sem limite máximo de frames.<br>
    <b>Stop</b>: interrompe a execução automática.<br>
    <b>Resetar incêndio</b>: zera o resfriamento acumulado, reinicia o frame e reposiciona o cursor.<br>
    <b>intervalo</b>: tempo aproximado entre frames, em milissegundos; valores menores aceleram a simulação, mas podem pesar mais no navegador.<br>
    <b>vel. borda</b>: deslocamento máximo do cursor quando ele percorre a trajetória de pontos calculados por offset de borda.<br>
    <b>vel. outras</b>: deslocamento máximo do cursor quando ele segue outras trajetórias, como focos fragmentados/centróides.<br>
    <b>permanência</b>: tempo, em frames, que o cursor permanece sobre um ponto após alcançá-lo antes de seguir para o próximo.<br>
    <b>thr quente</b>: limiar usado para construir a máscara filtrada das regiões quentes.<br>
    <b>erosão</b>: tamanho do elemento estruturante elíptico aplicado sobre a máscara binária.<br>
    <b>iter erosão</b>: número de repetições do ciclo erosão + dilatação.<br>
    <b>min pixels</b>: critério informativo de extinção, com faixa de 0 até {MAX_STOP_PIXELS} pixels. Ele não para a simulação automaticamente.<br>
    <b>n pontos</b>: quantidade de pontos da trajetória tangente à borda inferior na estratégia principal.<br>
    <b>offset borda</b>: deslocamento vertical dos pontos alguns pixels acima da borda inferior da região quente.<br>
    <b>thr parada</b>: limiar térmico usado para contar os pixels ainda quentes no status.<br>
    <b>prod bbox</b>: produto mínimo largura × altura para uma região ainda ser considerada contínua e manter a estratégia tangente à borda; abaixo disso, entra a estratégia por centróides fragmentados.<br>
    <b>raio atuação</b>: define a área de atuação do cursor redutor de chama; o círculo verde indica essa região.<br>
    <b>intensidade</b>: define a força do resfriamento aplicado pelo cursor.<br>
    <b>recuperação</b>: controla quão rápido a chama volta a se recompor/espalhar após o combate; valores baixos reduzem a recuperação, valores altos aumentam.<br>
    <b>setups</b>: salva o conjunto atual de parâmetros em JSON, recarrega setups pela lista e exclui setups salvos. O arquivo local fica em jet_automation/parameters_setups/user/firefighting_simulator/.<br>
    <b>cor dinâmica por área</b>: toda chama tem centro laranja e bordas amarelas; somente chamas grandes possuem núcleo vermelho, que diminui conforme a área total reduz.<br>
    <b>chuviscos visuais</b>: ruídos vermelho-alaranjados fixos poluem a imagem sem filtro e também podem aparecer na imagem final quando o filtro estiver pouco eficiente.<br>
    <b>renderização</b>: usa um único widget de imagem atualizado por frame, evitando recriar figuras Matplotlib e reduzindo o lag visual.
    </div>
    """
)


frame_box = widgets.IntText(value=0, description="frame", disabled=True)
interval_slider = widgets.IntSlider(value=160, min=80, max=700, step=10, description="intervalo", continuous_update=False)
vel_borda_slider = widgets.IntSlider(value=10, min=1, max=35, step=1, description="vel. borda", continuous_update=False)
vel_outras_slider = widgets.IntSlider(value=18, min=1, max=45, step=1, description="vel. outras", continuous_update=False)
dwell_slider = widgets.IntSlider(value=4, min=0, max=30, step=1, description="permanência", continuous_update=False)
thr_hot_slider = widgets.FloatSlider(value=0.58, min=0.00, max=1.00, step=0.01, description="thr quente", readout_format=".2f", continuous_update=False)
erosion_size_slider = widgets.IntSlider(value=3, min=1, max=11, step=2, description="erosão", continuous_update=False)
erosion_iters_slider = widgets.IntSlider(value=1, min=0, max=4, step=1, description="iter erosão", continuous_update=False)
stop_pixels_slider = widgets.IntSlider(value=60, min=0, max=MAX_STOP_PIXELS, step=5, description="min pixels", continuous_update=False)
n_points_slider = widgets.IntSlider(value=12, min=2, max=28, step=1, description="n pontos", continuous_update=False)
y_offset_slider = widgets.IntSlider(value=4, min=0, max=20, step=1, description="offset borda", continuous_update=False)
thr_stop_slider = widgets.FloatSlider(value=0.58, min=0.00, max=1.00, step=0.01, description="thr parada", readout_format=".2f", continuous_update=False)
bbox_prod_slider = widgets.IntSlider(value=3600, min=100, max=18000, step=100, description="prod bbox", continuous_update=False)
cool_radius_slider = widgets.IntSlider(value=70, min=15, max=120, step=1, description="raio atuação", continuous_update=False)
cool_intensity_slider = widgets.FloatSlider(value=1.00, min=0.20, max=2.00, step=0.05, description="intensidade", readout_format=".2f", continuous_update=False)
recovery_slider = widgets.FloatSlider(value=0.20, min=0.00, max=1.00, step=0.02, description="recuperação", readout_format=".2f", continuous_update=False)

start_button = widgets.Button(description="Iniciar", button_style="success", icon="play")
stop_button = widgets.Button(description="Stop", button_style="danger", icon="stop")
stop_button.disabled = True
reset_button = widgets.Button(description="Resetar incêndio", icon="refresh")

setup_name_text = widgets.Text(value="Meu setup", description="nome")
setup_dropdown = widgets.Dropdown(options=[], description="setups")
save_setup_button = widgets.Button(description="Salvar setup", button_style="info", icon="save")
load_setup_button = widgets.Button(description="Carregar", icon="upload")
delete_setup_button = widgets.Button(description="Excluir", button_style="danger", icon="trash")
setup_status = widgets.HTML(value="<i>Nenhum setup salvo ainda.</i>")

render_status = widgets.HTML(value="<b>Status:</b> aguardando início")
vis_image = widgets.Image(format="jpeg")

PARAM_WIDGETS = {
    "intervalo": interval_slider,
    "vel_borda": vel_borda_slider,
    "vel_outras": vel_outras_slider,
    "permanencia": dwell_slider,
    "thr_quente": thr_hot_slider,
    "erosao": erosion_size_slider,
    "iter_erosao": erosion_iters_slider,
    "min_pixels": stop_pixels_slider,
    "n_pontos": n_points_slider,
    "offset_borda": y_offset_slider,
    "thr_parada": thr_stop_slider,
    "prod_bbox": bbox_prod_slider,
    "raio_atuacao": cool_radius_slider,
    "intensidade_cursor": cool_intensity_slider,
    "recuperacao": recovery_slider,
}

SAVED_SETUPS = {}
# Caminhos portáteis do projeto BLAZE para setups do simulador
CURRENT_DIR = Path.cwd().resolve()

for _candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
    if (_candidate / 'src' / 'blaze_paths.py').exists():
        PROJECT_ROOT = _candidate
        break
else:
    raise RuntimeError(
        'Não foi possível localizar a raiz do projeto BLAZE. '
        'Abra este notebook a partir de uma pasta dentro do repositório BLAZE.'
    )

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from blaze_paths import (
    JET_OFFICIAL_SETUPS_DIR,
    JET_USER_SETUPS_DIR,
    JET_RESULTS_DIR,
    ensure_base_dirs,
)

ensure_base_dirs()

OFFICIAL_SETUP_FILE = JET_OFFICIAL_SETUPS_DIR / 'firefighting_simulator' / 'setups_simulacao_incendio.json'
USER_SETUP_FILE = JET_USER_SETUPS_DIR / 'firefighting_simulator' / 'setups_simulacao_incendio_user.json'

USER_SETUP_FILE.parent.mkdir(parents=True, exist_ok=True)

if not USER_SETUP_FILE.exists() and OFFICIAL_SETUP_FILE.exists():
    shutil.copy2(OFFICIAL_SETUP_FILE, USER_SETUP_FILE)
    print(f'Arquivo oficial de setups copiado para edição local: {USER_SETUP_FILE}')

SETUP_FILE = USER_SETUP_FILE



def capturar_setup_atual():
    return {nome: widget.value for nome, widget in PARAM_WIDGETS.items()}


def normalizar_setup(setup):
    """Mantém somente chaves conhecidas e ignora parâmetros antigos removidos."""
    if not isinstance(setup, dict):
        return {}
    return {chave: valor for chave, valor in setup.items() if chave in PARAM_WIDGETS}


def carregar_setups_do_arquivo():
    global SAVED_SETUPS
    if not os.path.exists(SETUP_FILE):
        SAVED_SETUPS = {}
        return
    try:
        with open(SETUP_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, dict):
            SAVED_SETUPS = {}
            return
        SAVED_SETUPS = {str(nome): normalizar_setup(setup) for nome, setup in data.items()}
        SAVED_SETUPS = {nome: setup for nome, setup in SAVED_SETUPS.items() if setup}
    except Exception as e:
        SAVED_SETUPS = {}
        setup_status.value = f"<span style='color:#cc4444'><b>Não foi possível ler o arquivo de setups:</b> {e}</span>"


def salvar_setups_no_arquivo():
    try:
        with open(SETUP_FILE, "w", encoding="utf-8") as f:
            json.dump(SAVED_SETUPS, f, ensure_ascii=False, indent=2)
        return True, None
    except Exception as e:
        return False, e


def refresh_setup_dropdown(prefer=None):
    nomes = sorted(SAVED_SETUPS.keys())
    if not nomes:
        setup_dropdown.options = []
        setup_dropdown.value = None
        return
    setup_dropdown.options = nomes
    if prefer in SAVED_SETUPS:
        setup_dropdown.value = prefer
    elif setup_dropdown.value not in SAVED_SETUPS:
        setup_dropdown.value = nomes[0]


def aplicar_setup_por_nome(nome, atualizar_tela=True):
    if nome not in SAVED_SETUPS:
        setup_status.value = f"<span style='color:#cc4444'><b>Setup não encontrado:</b> {nome}</span>"
        return
    setup = normalizar_setup(SAVED_SETUPS[nome])
    for chave, valor in setup.items():
        if chave in PARAM_WIDGETS:
            PARAM_WIDGETS[chave].value = valor
    setup_name_text.value = nome
    setup_dropdown.value = nome
    setup_status.value = f"<span style='color:#44aa44'><b>Setup carregado:</b> {nome}</span>"
    if atualizar_tela:
        atualizar()


def salvar_setup(_=None):
    nome = setup_name_text.value.strip()
    if not nome:
        setup_status.value = "<span style='color:#cc4444'><b>Informe um nome válido para salvar o setup.</b></span>"
        return
    existed = nome in SAVED_SETUPS
    SAVED_SETUPS[nome] = capturar_setup_atual()
    ok, erro = salvar_setups_no_arquivo()
    refresh_setup_dropdown(prefer=nome)
    if not ok:
        setup_status.value = f"<span style='color:#cc4444'><b>Setup mantido na sessão, mas não salvo em arquivo:</b> {erro}</span>"
    elif existed:
        setup_status.value = f"<span style='color:#dd8800'><b>Setup sobrescrito e salvo em arquivo:</b> {nome}</span>"
    else:
        setup_status.value = f"<span style='color:#44aa44'><b>Setup salvo em arquivo:</b> {nome}</span>"


def carregar_setup(_=None):
    nome = setup_dropdown.value
    if nome is None:
        setup_status.value = "<span style='color:#cc4444'><b>Selecione um setup para carregar.</b></span>"
        return
    aplicar_setup_por_nome(nome, atualizar_tela=True)


def excluir_setup(_=None):
    nome = setup_dropdown.value
    if nome is None:
        setup_status.value = "<span style='color:#cc4444'><b>Selecione um setup para excluir.</b></span>"
        return
    SAVED_SETUPS.pop(nome, None)
    ok, erro = salvar_setups_no_arquivo()
    refresh_setup_dropdown()
    setup_name_text.value = "Meu setup"
    if ok:
        setup_status.value = f"<span style='color:#cc4444'><b>Setup excluído do arquivo:</b> {nome}</span>"
    else:
        setup_status.value = f"<span style='color:#cc4444'><b>Setup excluído da sessão, mas houve erro ao atualizar o arquivo:</b> {erro}</span>"


def setup_inicial():
    carregar_setups_do_arquivo()
    if "Padrão" not in SAVED_SETUPS:
        SAVED_SETUPS["Padrão"] = capturar_setup_atual()
        salvar_setups_no_arquivo()
    refresh_setup_dropdown(prefer="Padrão")
    setup_name_text.value = "Padrão"
    setup_status.value = f"<span style='color:#44aa44'><b>Setups carregados:</b> {len(SAVED_SETUPS)} | arquivo: {SETUP_FILE}</span>"


def aplicar_layout_compacto():
    """Ajusta a interface para duas caixas por linha, com sliders legíveis."""
    slider_layout = widgets.Layout(width="300px", margin="0 6px 4px 0")
    slider_style = {"description_width": "112px"}

    slider_widgets = [
        interval_slider, vel_borda_slider, vel_outras_slider, dwell_slider,
        thr_hot_slider, erosion_size_slider, erosion_iters_slider,
        stop_pixels_slider, n_points_slider, y_offset_slider, thr_stop_slider,
        bbox_prod_slider, cool_radius_slider, cool_intensity_slider, recovery_slider,
    ]
    for widget in slider_widgets:
        widget.layout = slider_layout
        widget.style = slider_style

    frame_box.layout = widgets.Layout(width="170px", margin="0 6px 4px 0")
    frame_box.style = {"description_width": "55px"}

    setup_name_text.layout = widgets.Layout(width="300px", margin="0 6px 4px 0")
    setup_name_text.style = {"description_width": "55px"}
    setup_dropdown.layout = widgets.Layout(width="300px", margin="0 6px 4px 0")
    setup_dropdown.style = {"description_width": "65px"}

    for botao in [start_button, stop_button, reset_button, save_setup_button, load_setup_button, delete_setup_button]:
        botao.layout = widgets.Layout(width="auto", margin="0 6px 4px 0")

    desc.layout = widgets.Layout(width="100%", max_width="1280px", margin="0 0 6px 0")
    render_status.layout = widgets.Layout(width="100%", max_width="1280px", margin="2px 0 4px 0")
    vis_image.layout = widgets.Layout(width="100%", max_width="1280px", margin="4px 0 0 0")


def linha_compacta(filhos):
    return widgets.HBox(
        list(filhos),
        layout=widgets.Layout(
            display="flex",
            flex_flow="row wrap",
            align_items="center",
            width="100%",
            margin="0"
        )
    )


def criar_caixa(titulo, filhos, largura="632px"):
    header = widgets.HTML(
        value=f"<b>{titulo}</b>",
        layout=widgets.Layout(margin="0 0 3px 0", height="20px")
    )
    box = widgets.VBox([header] + list(filhos))
    box.layout = widgets.Layout(
        border="1px solid #555",
        padding="6px 7px 5px 7px",
        margin="3px 5px 5px 0",
        width=largura,
        min_width=largura,
        max_width=largura,
        height="auto"
    )
    return box


def criar_linha_duas_caixas(caixa_esq, caixa_dir):
    return widgets.HBox(
        [caixa_esq, caixa_dir],
        layout=widgets.Layout(
            display="flex",
            flex_flow="row nowrap",
            align_items="stretch",
            width="100%",
            margin="0"
        )
    )


def atualizar(*args):
    frame_box.value = int(state["frame"])
    desenhar(
        frame=int(state["frame"]),
        vel_borda=int(vel_borda_slider.value),
        vel_outras=int(vel_outras_slider.value),
        dwell_frames=int(dwell_slider.value),
        thr_hot=float(thr_hot_slider.value),
        erosion_size=int(erosion_size_slider.value),
        erosion_iters=int(erosion_iters_slider.value),
        stop_pixels=int(stop_pixels_slider.value),
        n_points=int(n_points_slider.value),
        thr_stop=float(thr_stop_slider.value),
        y_offset=int(y_offset_slider.value),
        bbox_prod_min=int(bbox_prod_slider.value),
        cool_radius=int(cool_radius_slider.value),
        cool_intensity=float(cool_intensity_slider.value),
        recovery_rate=float(recovery_slider.value),
    )


async def loop_execucao_continua():
    while state["running"]:
        state["frame"] += 1
        atualizar()
        await asyncio.sleep(max(0.08, float(interval_slider.value) / 1000.0))

    start_button.disabled = False
    stop_button.disabled = True


def iniciar_execucao(_=None):
    if state["running"]:
        return
    state["running"] = True
    start_button.disabled = True
    stop_button.disabled = False
    state["animation_task"] = asyncio.ensure_future(loop_execucao_continua())


def parar_execucao(_=None):
    state["running"] = False
    start_button.disabled = False
    stop_button.disabled = True
    atualizar()


def reset_state(_=None):
    cool_gray[:] = 0
    cool_yellow[:] = 0
    state["frame"] = 0
    frame_box.value = 0
    last_frame["value"] = 0
    state["x"] = 26
    state["y"] = H - 24
    state["last_move_frame"] = 0
    state["pause_until_frame"] = -1
    state["route"] = []
    state["route_index"] = 0
    state["mode"] = "borda"
    state["movement_phase"] = "aproximação/entre trajetórias"
    state["is_extinguished"] = False
    state["main_region"] = None
    state["secondary_region"] = None
    state["regions"] = []
    state["hot_pixels"] = 0
    atualizar()


start_button.on_click(iniciar_execucao)
stop_button.on_click(parar_execucao)
reset_button.on_click(reset_state)
save_setup_button.on_click(salvar_setup)
load_setup_button.on_click(carregar_setup)
delete_setup_button.on_click(excluir_setup)

for w in [interval_slider, vel_borda_slider, vel_outras_slider, dwell_slider, thr_hot_slider, erosion_size_slider,
          erosion_iters_slider, stop_pixels_slider, n_points_slider, y_offset_slider,
          thr_stop_slider, bbox_prod_slider, cool_radius_slider, cool_intensity_slider, recovery_slider]:
    w.observe(atualizar, names="value")

aplicar_layout_compacto()

controle_execucao_box = criar_caixa("Execução", [
    linha_compacta([start_button, stop_button, reset_button, frame_box]),
    linha_compacta([interval_slider]),
])

velocidade_box = criar_caixa("Velocidade e permanência", [
    linha_compacta([vel_borda_slider, vel_outras_slider]),
    linha_compacta([dwell_slider]),
])

filtro_box = criar_caixa("Filtro", [
    linha_compacta([thr_hot_slider, erosion_size_slider]),
    linha_compacta([erosion_iters_slider]),
])

criterio_box = criar_caixa("Critério de parada e estratégia", [
    linha_compacta([stop_pixels_slider, thr_stop_slider]),
    linha_compacta([n_points_slider, y_offset_slider]),
    linha_compacta([bbox_prod_slider]),
])

atuacao_chama_box = criar_caixa("Atuação do cursor e comportamento da chama", [
    linha_compacta([cool_radius_slider, cool_intensity_slider]),
    linha_compacta([recovery_slider]),
])

setups_box = criar_caixa("Setups paramétricos", [
    linha_compacta([setup_name_text]),
    linha_compacta([setup_dropdown]),
    linha_compacta([save_setup_button, load_setup_button, delete_setup_button]),
    setup_status,
])

ui = widgets.VBox([
    desc,
    criar_linha_duas_caixas(controle_execucao_box, velocidade_box),
    criar_linha_duas_caixas(filtro_box, criterio_box),
    criar_linha_duas_caixas(atuacao_chama_box, setups_box),
    render_status,
    vis_image
], layout=widgets.Layout(width="100%", max_width="1280px", align_items="stretch"))

display(ui)
setup_inicial()
reset_state()
